# Tree Models
Random forest (RF) and gradient boosted regression trees (GBRT)

## Panel R²oos (monthly, all stocks) 

In [1]:
import pandas as pd
import numpy as np
import gc
import os
import shutil
import warnings
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from itertools import product
from tqdm.auto import tqdm

In [2]:
# ==========================================
# 1. DATA UPLOAD / LOAD PANEL
# ==========================================
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# This mirrors the Drive-to-local-runtime pattern from the NN notebook.
# To use the full 936-feature panel instead, change this filename/path in Drive.
DRIVE_FILE = Path('/content/drive/MyDrive/Replication Paper/Data/gkx2020_panel_trimmed.parquet')
LOCAL_COLAB_DATA_DIR = Path('/content/data')
LOCAL_FALLBACK_FILES = [
    Path('data/gkx2020_panel_trimmed.parquet'),
    Path('data/gkx2020_panel.parquet'),
]
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Replication Paper/Data/Results/tree_results')

# 2. Load the data
print("Loading data...")

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_DIR = LOCAL_COLAB_DATA_DIR
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if not DRIVE_FILE.exists():
        raise FileNotFoundError(f"File not found in Google Drive: {DRIVE_FILE}")

    LOCAL_FILE = DATA_DIR / DRIVE_FILE.name
    if not LOCAL_FILE.exists():
        print("Copying file from Google Drive to local Colab runtime...")
        shutil.copy2(DRIVE_FILE, LOCAL_FILE)
        print("Copy complete.")
    else:
        print("Local copy already exists. Skipping copy.")

    if SAVE_OUTPUTS_TO_DRIVE:
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_DIR = Path('data')
    LOCAL_FILE = next((path for path in LOCAL_FALLBACK_FILES if path.exists()), None)
    if LOCAL_FILE is None:
        searched = ', '.join(str(path) for path in LOCAL_FALLBACK_FILES)
        raise FileNotFoundError(f"Could not find a local panel parquet. Searched: {searched}")

print(f"Reading panel from: {LOCAL_FILE}")
df = pd.read_parquet(LOCAL_FILE)

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
panel_memory_gb = df.memory_usage(deep=True).sum() / 1024**3
panel_dates = df['date'].dt.to_period('M')

print(f"Panel shape: {df.shape}")
print(f"Date range: {panel_dates.min()} to {panel_dates.max()}")
print(f"Unique stocks: {df['permno'].nunique():,}")
print(f"Memory usage: {panel_memory_gb:.2f} GB")

# Define features
cols_to_drop = ['permno', 'date', 'exchcd', 'ret_excess', 'split', 'year', 'me', 'ret', 'prc']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

Mounted at /content/drive
Copying file from Google Drive to local Colab runtime...
Copy complete.
Reading panel from: /content/data/gkx2020_panel_trimmed.parquet
Panel shape: (3305648, 284)
Date range: 1957-03 to 2021-12
Unique stocks: 25,770
Memory usage: 2.74 GB


In [ ]:
# ==========================================
# 2. MODEL CONFIGURATION, TRAINING, AND OUTPUTS
# ==========================================
def detect_cuda():
    """Detect a CUDA GPU without making PyTorch a hard dependency."""
    try:
        import torch
        if torch.cuda.is_available():
            return True, torch.cuda.get_device_name(0)
    except Exception:
        pass

    if Path('/proc/driver/nvidia/version').exists():
        return True, 'NVIDIA GPU detected'

    return False, 'not detected'


CUDA_AVAILABLE, CUDA_DEVICE_NAME = detect_cuda()
XGB_USE_CUDA = CUDA_AVAILABLE


def xgb_major_version() -> int:
    major = xgb.__version__.split('.', 1)[0]
    digits = ''.join(ch for ch in major if ch.isdigit())
    return int(digits) if digits else 0


def xgb_tree_runtime_params():
    """Return XGBoost tree runtime parameters, using CUDA when Colab exposes a GPU."""
    if XGB_USE_CUDA:
        if xgb_major_version() >= 2:
            return {'tree_method': 'hist', 'device': 'cuda'}
        return {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'}

    return {'tree_method': 'hist'}


def to_xgb_array(data):
    """Return data on the same device as the XGBoost booster to avoid predict fallback warnings."""
    if not XGB_USE_CUDA:
        return data

    try:
        import cupy as cp
    except ImportError as exc:
        raise ImportError(
            "CUDA XGBoost prediction without device-mismatch warnings requires CuPy. "
            "In Colab, install a CUDA-matched CuPy build or switch XGB_USE_CUDA = False."
        ) from exc

    if isinstance(data, (pd.DataFrame, pd.Series)):
        array = data.to_numpy(dtype=np.float32, copy=False)
    else:
        array = np.asarray(data, dtype=np.float32)

    return cp.asarray(np.ascontiguousarray(array))


def xgb_predict(model, X):
    """Predict with XGBoost and return a NumPy array for downstream pandas/sklearn code."""
    preds = model.predict(X)

    if XGB_USE_CUDA:
        import cupy as cp
        if isinstance(preds, cp.ndarray):
            return cp.asnumpy(preds)

    return np.asarray(preds)


def clear_xgb_device_cache():
    """Release cached CuPy blocks between annual refits when CUDA is active."""
    if not XGB_USE_CUDA:
        return

    try:
        import cupy as cp
    except ImportError:
        return

    cp.get_default_memory_pool().free_all_blocks()


def max_features_to_colsample_bynode(max_features, n_features):
    """Map sklearn-style RF max_features to XGBoost's colsample_bynode fraction."""
    min_fraction = 1.0 / n_features

    if max_features is None:
        return 1.0
    if isinstance(max_features, str):
        if max_features == 'sqrt':
            return max(min_fraction, 1.0 / np.sqrt(n_features))
        if max_features == 'log2':
            return max(min_fraction, np.log2(n_features) / n_features)
        raise ValueError(f"Unsupported max_features value for XGBoost RF: {max_features!r}")
    if isinstance(max_features, (int, np.integer)):
        return min(1.0, max(min_fraction, float(max_features) / n_features))
    if isinstance(max_features, (float, np.floating)):
        return min(1.0, max(min_fraction, float(max_features)))

    raise TypeError(f"Unsupported max_features type: {type(max_features)!r}")


def make_rf_model(params, n_estimators, n_features):
    """Create an XGBoost random forest, using CUDA when available."""
    rf_params = dict(params)
    max_features = rf_params.pop('max_features', None)
    rf_params['colsample_bynode'] = max_features_to_colsample_bynode(max_features, n_features)

    model_params = {
        'objective': 'reg:squarederror',
        'n_estimators': n_estimators,
        'learning_rate': 1.0,
        'subsample': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        **xgb_tree_runtime_params(),
        **rf_params,
    }

    return xgb.XGBRFRegressor(**model_params)


def make_gbrt_model(params, use_early_stopping=False, n_estimators=1000):
    model_params = {
        'objective': 'reg:pseudohubererror',
        'random_state': 42,
        'n_estimators': n_estimators,   # upper bound during tuning, fixed count for final refit
        **xgb_tree_runtime_params(),
        **params,
    }

    if use_early_stopping:
        model_params['early_stopping_rounds'] = 20

    return xgb.XGBRegressor(**model_params)


print(f"Running in Colab: {IN_COLAB}")
print(f"CUDA available for XGBoost: {CUDA_AVAILABLE} ({CUDA_DEVICE_NAME})")
print(f"XGBoost version: {xgb.__version__}")
print("Note: RF and GBRT both use XGBoost; both train on CUDA when available.")

# ==========================================
# 1. SUPERCOMPUTER MASTER TOGGLE
# ==========================================
# Set to False for local testing. Set to True for the final 48-hour run.
SUPERCOMPUTER_MODE = True

if SUPERCOMPUTER_MODE:
    print("WARNING: Supercomputer mode enabled. This will take a long time.")
    TEST_YEARS = range(1987, 2022)  # Full GKX-style test period, inclusive through 2021
    VAL_WINDOW_YEARS = 12           # The full GKX validation window
    TUNE_ESTIMATORS = 300           # Build robust models during tuning
    FINAL_ESTIMATORS = 300

    # The Full GKX Grid Search
    # Note: max_features of 30, 50, 100 are good choices when you switch back to the 920-column dataset
    rf_grid = {
        'max_depth': [1, 2, 3, 4, 5, 6],
        'max_features': ['sqrt', 30, 50, 100]
    }
    gbrt_grid = {
        'max_depth': [1, 2, 3, 4, 5, 6],
        'learning_rate': [0.01, 0.05, 0.1],
    }

else:
    print("Local Test Mode enabled. Using restricted parameters.")
    TEST_YEARS = [1987, 1988]       # Predict just 2 years
    VAL_WINDOW_YEARS = 2            # Only look back 2 years for validation
    TUNE_ESTIMATORS = 50            # Build shallow models to speed up Grid Search
    FINAL_ESTIMATORS = 300

    # The "Speed-Test" Grid Search (Bare minimum to ensure code works)
    rf_grid = {
        'max_depth': [3, 4],
        'max_features': ['sqrt']
    }
    gbrt_grid = {
        'max_depth': [2, 3],
        'learning_rate': [0.1],
        'n_estimators': [300]  # Let the model choose how many trees to build!
    }
# ==========================================

# 3. Setup Hyperparameter Grids (Variables now fed from Toggle block)
def make_grid(param_dict):
    keys = param_dict.keys()
    vals = param_dict.values()
    return [dict(zip(keys, v)) for v in product(*vals)]

rf_params_list = make_grid(rf_grid)
gbrt_params_list = make_grid(gbrt_grid)

all_predictions = []
YEARLY_PREDS_DIR = DATA_DIR / 'yearly_preds'
YEARLY_PREDS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PREDS_FILE = DATA_DIR / 'TEST_recursive_predictions.parquet'

# 4. GKX OOS R-Squared Formula
def calculate_oos_r2(y_true, y_pred):
    denominator = (y_true ** 2).sum()  # Baseline is predicting zero
    numerator = ((y_true - y_pred) ** 2).sum()
    return 1 - (numerator / denominator)

# 5. THE RECURSIVE REFITTING LOOP
print(f"Starting execution for test years: {TEST_YEARS}")
for test_year in tqdm(TEST_YEARS, desc="Years Processed"):

    # Define time windows for this specific loop
    val_end_year = test_year - 1
    val_start_year = test_year - VAL_WINDOW_YEARS

    # Create masks
    train_mask = df['year'] < val_start_year
    val_mask = (df['year'] >= val_start_year) & (df['year'] <= val_end_year)
    test_mask = df['year'] == test_year

    if test_mask.sum() == 0:
        continue

    # Extract sets. Convert to float32 to save 50% RAM instantly
    X_train = df.loc[train_mask, feature_cols].fillna(0).astype('float32')
    y_train = df.loc[train_mask, 'ret_excess'].astype('float32')

    X_val = df.loc[val_mask, feature_cols].fillna(0).astype('float32')
    y_val = df.loc[val_mask, 'ret_excess'].astype('float32')

    X_test = df.loc[test_mask, feature_cols].fillna(0).astype('float32')
    y_test = df.loc[test_mask, 'ret_excess'].astype('float32')

    # Put XGBoost inputs on the same device as the booster. This avoids the
    # "mismatched devices" fallback warning during validation/test prediction.
    X_train_xgb = to_xgb_array(X_train)
    y_train_xgb = to_xgb_array(y_train)
    X_val_xgb = to_xgb_array(X_val)
    y_val_xgb = to_xgb_array(y_val)
    X_test_xgb = to_xgb_array(X_test)

    # --- A. HYPERPARAMETER TUNING (Validation Set) ---
    print(f"\n[{test_year}] Tuning Random Forest...")
    best_rf_val_mse = float('inf')
    best_rf_params = None

    for params in rf_params_list:
        model = make_rf_model(params, TUNE_ESTIMATORS, len(feature_cols))
        model.fit(X_train_xgb, y_train_xgb)
        val_preds = xgb_predict(model, X_val_xgb)
        mse = mean_squared_error(y_val, val_preds)

        if mse < best_rf_val_mse:
            best_rf_val_mse = mse
            best_rf_params = params

    print(f"[{test_year}] Best RF Params: {best_rf_params}")

    print(f"[{test_year}] Tuning GBRT...")
    best_gbrt_val_mse = float('inf')
    best_gbrt_params = None
    best_gbrt_n_estimators = 300

    for params in gbrt_params_list:
        model = make_gbrt_model(params, use_early_stopping=True)
        model.fit(X_train_xgb, y_train_xgb, eval_set=[(X_val_xgb, y_val_xgb)], verbose=False)
        val_preds = xgb_predict(model, X_val_xgb)
        mse = mean_squared_error(y_val, val_preds)

        if mse < best_gbrt_val_mse:
            best_gbrt_val_mse = mse
            best_gbrt_params = params
            best_gbrt_n_estimators = int(model.best_iteration) + 1

    print(f"[{test_year}] Best GBRT Params: {best_gbrt_params}")
    print(f"[{test_year}] Best GBRT n_estimators: {best_gbrt_n_estimators}")

    # --- B. FIT FINAL MODELS ---
    # Merge Train and Val together to give the final model the most recent data
    X_train_full = pd.concat([X_train, X_val])
    y_train_full = pd.concat([y_train, y_val])

    # Free up validation GPU arrays before fitting massive final models.
    del X_train_xgb, y_train_xgb, X_val_xgb, y_val_xgb
    clear_xgb_device_cache()

    X_train_full_xgb = to_xgb_array(X_train_full)
    y_train_full_xgb = to_xgb_array(y_train_full)

    # Free up CPU split frames before fitting final models.
    del X_train, y_train, X_val, y_val, model
    gc.collect()

    print(f"[{test_year}] Fitting final models and predicting out-of-sample...")
    # Final RF
    final_rf = make_rf_model(best_rf_params, FINAL_ESTIMATORS, len(feature_cols))
    final_rf.fit(X_train_full_xgb, y_train_full_xgb)
    rf_test_preds = xgb_predict(final_rf, X_test_xgb)

    # Final GBRT
    final_gbrt = make_gbrt_model(best_gbrt_params, n_estimators=best_gbrt_n_estimators)
    final_gbrt.fit(X_train_full_xgb, y_train_full_xgb)
    gbrt_test_preds = xgb_predict(final_gbrt, X_test_xgb)

    # --- C. SAVE RESULTS ---
    # UPDATED: Added permno and mvel1 so we can rank stocks later
    yearly_results = pd.DataFrame({
        'permno': df.loc[test_mask, 'permno'],
        'date': df.loc[test_mask, 'date'],
        'mvel1': df.loc[test_mask, 'mvel1'],
        'actual_return': y_test,
        'rf_predicted': rf_test_preds,
        'gbrt_predicted': gbrt_test_preds
    })

    all_predictions.append(yearly_results)
    yearly_pred_file = YEARLY_PREDS_DIR / f"TEST_preds_{test_year}.parquet"
    yearly_results.to_parquet(yearly_pred_file)

    if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
        yearly_results.to_parquet(DRIVE_OUTPUT_DIR / yearly_pred_file.name)

    # Aggressive garbage collection to prevent memory crashes across loops
    del X_train_full, y_train_full, X_train_full_xgb, y_train_full_xgb, X_test, X_test_xgb, y_test, yearly_results
    gc.collect()
    clear_xgb_device_cache()

# 6. Final Evaluation
print("\nStitching predictions together...")
final_predictions_df = pd.concat(all_predictions, ignore_index=True)
final_predictions_df.to_parquet(FINAL_PREDS_FILE)

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    combined_out_path = DRIVE_OUTPUT_DIR / FINAL_PREDS_FILE.name
    final_predictions_df.to_parquet(combined_out_path, index=False)
    print(f"Saved combined tree-model predictions to {combined_out_path}")

    # ====================================================
    # Save OOS predictions for each tree model
    # ====================================================
    tree_prediction_specs = {
        "RF": "rf_predicted",
        "GBRT": "gbrt_predicted",
    }

    for model_name, pred_col in tree_prediction_specs.items():
        out_df = pd.DataFrame({
            "test_year": final_predictions_df["date"].dt.year,
            "date": final_predictions_df["date"],
            "permno": final_predictions_df["permno"],
            "actual": final_predictions_df["actual_return"],
            "predicted": final_predictions_df[pred_col],
        })
        out_path = DRIVE_OUTPUT_DIR / f"{model_name}_predictions.parquet"
        out_df.to_parquet(out_path, index=False)
        print(f"Saved {model_name} predictions to {out_path}")

    print("\nDone! All tree-model predictions saved.")

true_rf_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df['rf_predicted'])
true_gbrt_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df['gbrt_predicted'])

print("\n==========================================")
print(f"TEST RUN - Random Forest R2_OOS: {true_rf_r2 * 100:.3f}%")
print(f"TEST RUN - Gradient Boosting (XGBoost) R2_OOS: {true_gbrt_r2 * 100:.3f}%")
print("==========================================")

Running in Colab: True
CUDA available for XGBoost: True (NVIDIA A100-SXM4-40GB)
XGBoost version: 3.2.0
Note: RF and GBRT both use XGBoost; both train on CUDA when available.
Starting execution for test years: range(1987, 2022)


Years Processed:   0%|          | 0/35 [00:00<?, ?it/s]


[1987] Tuning Random Forest...
[1987] Best RF Params: {'max_depth': 1, 'max_features': 'sqrt'}
[1987] Tuning GBRT...
[1987] Best GBRT Params: {'max_depth': 2, 'learning_rate': 0.01}
[1987] Best GBRT n_estimators: 15
[1987] Fitting final models and predicting out-of-sample...

[1988] Tuning Random Forest...
[1988] Best RF Params: {'max_depth': 1, 'max_features': 'sqrt'}
[1988] Tuning GBRT...
[1988] Best GBRT Params: {'max_depth': 5, 'learning_rate': 0.01}
[1988] Best GBRT n_estimators: 5
[1988] Fitting final models and predicting out-of-sample...

[1989] Tuning Random Forest...
[1989] Best RF Params: {'max_depth': 1, 'max_features': 'sqrt'}
[1989] Tuning GBRT...
[1989] Best GBRT Params: {'max_depth': 5, 'learning_rate': 0.01}
[1989] Best GBRT n_estimators: 5
[1989] Fitting final models and predicting out-of-sample...

[1990] Tuning Random Forest...
[1990] Best RF Params: {'max_depth': 3, 'max_features': 'sqrt'}
[1990] Tuning GBRT...
[1990] Best GBRT Params: {'max_depth': 2, 'learning_r

# To check data leakage

In [ ]:
import pandas as pd

print("\n--- QUICK HUNT FOR DATA LEAKAGE ---")
# 1. Grab just the first 50,000 rows so this runs in 3 seconds
sample_df = df.head(50000).fillna(0)

# 2. Separate features and target
X_sample = sample_df[feature_cols]
y_sample = sample_df['ret_excess']

# 3. Train a quick diagnostic XGBoost Random Forest
print("Training diagnostic model...")
diagnostic_rf = make_rf_model({'max_depth': 5, 'max_features': 'sqrt'}, n_estimators=50, n_features=len(feature_cols))
diagnostic_rf.fit(to_xgb_array(X_sample), to_xgb_array(y_sample))

# 4. Print the top 10 features it relied on
importances = pd.Series(diagnostic_rf.feature_importances_, index=feature_cols)
print("\nTop 10 Most Important Features:")
print(importances.sort_values(ascending=False).head(10))

##  Sub sample R²oos (monthly, top and bottom 1000) 

In [ ]:
# ==========================================
# Deliverable 2: Subsample R²oos (Top 1000 & Bottom 1000)
# ==========================================

print("--- DELIVERABLE 2: Subsample R²oos ---")

# 1. Sort predictions by date, and then by market cap (mvel1) in descending order.
# Dropping NaNs in mvel1 ensures we only rank stocks with valid size data.
sorted_preds = final_predictions_df.dropna(subset=['mvel1']).sort_values(
    by=['date', 'mvel1', 'permno'], 
    ascending=[True, False, True] # False for mvel1 (Largest first), True for permno
)

# 2. Extract the Top 1000 (Largest) and Bottom 1000 (Smallest) stocks per month
top_1000_df = sorted_preds.groupby('date').head(1000)
bottom_1000_df = sorted_preds.groupby('date').tail(1000)

print(f"Total observations in Top 1000 subset: {len(top_1000_df):,}")
print(f"Total observations in Bottom 1000 subset: {len(bottom_1000_df):,}\n")

# 3. Calculate R²oos for the Top 1000
# Note: Re-using the 'calculate_oos_r2' function you defined earlier in the notebook.
top_1000_rf_r2 = calculate_oos_r2(top_1000_df['actual_return'], top_1000_df['rf_predicted'])
top_1000_gbrt_r2 = calculate_oos_r2(top_1000_df['actual_return'], top_1000_df['gbrt_predicted'])

# 4. Calculate R²oos for the Bottom 1000
bot_1000_rf_r2 = calculate_oos_r2(bottom_1000_df['actual_return'], bottom_1000_df['rf_predicted'])
bot_1000_gbrt_r2 = calculate_oos_r2(bottom_1000_df['actual_return'], bottom_1000_df['gbrt_predicted'])

# 5. Display the final metrics (Updated Labels)
print("Top 1000 Largest Stocks (by mvel1):")
print(f"  Random Forest R²oos:   {top_1000_rf_r2 * 100:.3f}%")
print(f"  GBRT + H R²oos:        {top_1000_gbrt_r2 * 100:.3f}%\n")

print("Bottom 1000 Smallest Stocks (by mvel1):")
print(f"  Random Forest R²oos:   {bot_1000_rf_r2 * 100:.3f}%")
print(f"  GBRT + H R²oos:        {bot_1000_gbrt_r2 * 100:.3f}%")

## Annual R²oos- Table 2


In [ ]:
import pandas as pd

# 1. Define the R2_oos calculation function
def calc_r2_oos(df, true_col, pred_col):
    y_true = df[true_col]
    y_pred = df[pred_col]
    
    # GKX (2020) out-of-sample R2 formula (baseline is 0)
    denominator = (y_true ** 2).sum()
    numerator = ((y_true - y_pred) ** 2).sum()
    
    return 1 - (numerator / denominator)

# Make sure the 'year' column exists
final_predictions_df['year'] = final_predictions_df['date'].dt.year

# 2. Group by year and calculate the year-by-year scores
annual_r2_rf = final_predictions_df.groupby('year').apply(
    lambda x: calc_r2_oos(x, 'actual_return', 'rf_predicted'),
    include_groups=False
)

annual_r2_gbrt = final_predictions_df.groupby('year').apply(
    lambda x: calc_r2_oos(x, 'actual_return', 'gbrt_predicted'),
    include_groups=False
)

# 3. Take the SIMPLE AVERAGE across all testing years and convert to percentage
avg_annual_rf = annual_r2_rf.mean() * 100
avg_annual_gbrt = annual_r2_gbrt.mean() * 100

# 4. Create the final formatted Table 2 to match the paper
table_2_final = pd.DataFrame({
    'Model': ['RF', 'GBRT'],
    'Annual R2_oos (%)': [avg_annual_rf, avg_annual_gbrt]
})

print("--- Table 2: Annual Out-of-Sample Predictive R2 ---")
print(table_2_final.to_string(index=False))

# Monthly prediction error series for DM tests 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Ensure date is a proper datetime format (if not already)
final_predictions_df['date'] = pd.to_datetime(final_predictions_df['date'])

# 2. Calculate squared errors for each observation
# The benchmark prediction is exactly 0, so its error is just the actual return
se_bench = final_predictions_df['actual_return'] ** 2
se_rf = (final_predictions_df['actual_return'] - final_predictions_df['rf_predicted']) ** 2
se_gbrt = (final_predictions_df['actual_return'] - final_predictions_df['gbrt_predicted']) ** 2

# 3. Calculate the difference in squared errors (d_it)
# A positive number here means the model beat the benchmark!
final_predictions_df['d_rf'] = se_bench - se_rf
final_predictions_df['d_gbrt'] = se_bench - se_gbrt

# 4. Group by date (month) to get the cross-sectional average difference (d_t)
monthly_d = final_predictions_df.groupby('date')[['d_rf', 'd_gbrt']].mean()

# 5. Define a function to compute the DM statistic with Newey-West standard errors
def calculate_dm_stat(d_t_series, lags=6):
    model = sm.OLS(d_t_series, np.ones(len(d_t_series)))
    results = model.fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    
    # Use .iloc[0] instead of [0] to fix the FutureWarning!
    dm_stat = results.tvalues.iloc[0]
    p_value = results.pvalues.iloc[0]
    
    return dm_stat, p_value

# 6. Calculate DM stats for both models
dm_rf, p_rf = calculate_dm_stat(monthly_d['d_rf'].dropna())
dm_gbrt, p_gbrt = calculate_dm_stat(monthly_d['d_gbrt'].dropna())

# 7. Combine into a neat summary table
table_3_summary = pd.DataFrame({
    'Model': ['Random Forest', 'GBRT'],
    'DM Statistic': [dm_rf, dm_gbrt],
    'p-value': [p_rf, p_gbrt]
})

print("Table 3: Diebold-Mariano Tests")
print(table_3_summary.to_string(index=False))
monthly_d.to_csv("data/rf_gbrt_monthly_DM_series.csv")

# Variable importance of top 20 characteristics for rf and gbrt +H

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("--- DELIVERABLE 5: Figure 4 (Variable Importance) ---")

# ==========================================
# 1. Random Forest Importance
# ==========================================
# Extract importances from the final trained RF model in memory
rf_importances = final_rf.feature_importances_

# Normalize so they sum to 1
rf_importances = rf_importances / rf_importances.sum()

# Create a DataFrame and get the Top 20
rf_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_importances
}).sort_values(by='Importance', ascending=False)

top20_rf = rf_imp_df.head(20)

# ==========================================
# 2. GBRT (XGBoost) Importance
# ==========================================
# Extract importances directly from the final XGBoost model!
gbrt_importances = final_gbrt.feature_importances_

# Normalize so they sum to 1
gbrt_importances = gbrt_importances / gbrt_importances.sum()

# Create a DataFrame and get the Top 20
gbrt_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gbrt_importances
}).sort_values(by='Importance', ascending=False)

top20_gbrt = gbrt_imp_df.head(20)

# ==========================================
# 3. Plotting Figure 4
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Random Forest Plot
axes[0].barh(top20_rf['Feature'][::-1], top20_rf['Importance'][::-1], color='steelblue')
axes[0].set_title('Random Forest - Top 20 Characteristics', fontsize=14)
axes[0].set_xlabel('Normalized Importance', fontsize=12)
axes[0].grid(axis='x', linestyle='--', alpha=0.7)

# GBRT Plot
axes[1].barh(top20_gbrt['Feature'][::-1], top20_gbrt['Importance'][::-1], color='darkorange')
axes[1].set_title('GBRT (XGBoost +Huber) - Top 20', fontsize=14)
axes[1].set_xlabel('Normalized Importance', fontsize=12)
axes[1].grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Save the data to CSV for the final report
rf_imp_df.to_csv("data/rf_feature_importance.csv", index=False)
gbrt_imp_df.to_csv("data/gbrt_feature_importance.csv", index=False)